# Reverse mode Automatic Differentiation

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/odsl-team/block-course-mar26-ml/blob/main/tutorials/ex_lecture_6_ad.ipynb)

1. Computational graphs
2. NN: forward pass and gradients
3. Checking gradients in jax
4. **Bonus:** Set up the training loop for the model

Nicole Hartman

ODSL ML Block Course

16 Apr 2022 (revised 23 Mar 2026)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Computational graphs

**Motivating example:**  Let's code up the model we had worked out in lecture.

$$f(x,y,z) = (x+y) \cdot z$$

<span style="color:red">$$f_1(x,y) = q = x+y$$</span>

<span style="color:blue">$$f_2(q,z) = q \cdot z$$</span>

<img src="../figures/toy-ex.jpg" width=650/>

**To Do:** Implement `f1`, `f2` and `f` functions in numpy with both a forward and a backward mode.

In [ ]:
def f1(x,y, grad=None):
    '''
    f1(x,y) = x+y
    
    Inputs:
    - x,y: The inputs to the functoin
    - grad: if not None, this is the "upstream gradient"

    Outpus:
    - If (upstream) grad is passed, return f1(x,y), output_grad
    - Else, just the "normal" fct, return f1(x,y)
    '''
    
    out = 
    if grad:
        raise NotImplementedError
    

        # below... grad = df/dq
        dfdx = 
        dfdy = 
        return out, np.array([dfdx, dfdy])
    else:
        return out

In [ ]:
def f2(q,z, grad=None):
    '''
    f2(q,z) = q*z
    '''

    out = 
    if grad:   
        raise NotImplementedError
    else: 
        return out

In [ ]:
def f(x,y,z, grad=None):
    '''
    f(x,y,z) = f2(q,x)
    '''

    # Need to run the program 1x to get the values to eval the grad
    
    q = 
    out = 
    
    if grad is None:

        return out

    else: 

        

        '''
        Fill in
        '''
        
        return out, [dfdx, dfdy, dfdz]
    
    
        
        

Check the forward mode

In [ ]:
x,y,z = 5,-2,4
print(f(x,y,z))


**What about the derivative?**

In [ ]:
dfdf = 1
out, grad = f(x,y,z, grad=dfdf)

for xi, dfi in zip(['x','y','z'],grad):
    print(f'df/d{xi}= {dfi}')


## Step 2a: Multi-layer perceptron (forward pass)

Define a simple feed-forward NN that accepts a input matrix

$$X \in \mathbb{R}^{N \times d}$$

(where for the example just above $N=200$, $d=5$)

and constructs an MLP with a single hidden layer with $H = 16$ units, and a ReLU non-linearity.

**Hint:** this is just three equations:

$$z = X W_1^T + b_1$$





$$y_\textrm{pred} =h W_2^T  + b_2$$

where $W_1 \in \mathbb{R}^{H \times d}, b_1 \in \mathbb{R}^{H}, W_2 \in \mathbb{R}^{1 \times H}, b_2 \in \mathbb{R}$,
$z \in \mathbb{R}^{N\times H}$, $h \in \mathbb{R}^{N \times H}$

$$h = \mathrm{ReLU}(z_1)$$

**Potentially useful functions:** 
- `np.einsum`
- `np.where`
- `np.random.randn`

**Dataset**

As a motivating example, let's fit a 3rd order polynomial:

$$y = \theta_0 x^3 + \theta_1 x^2 + \theta_2 x + \theta_3,$$

with $\theta = [5, 4, -2, -0.7]$.

In [ ]:
coeffs_true = [5, 4, -2, -0.7]

def generate_data(N):
    x = np.random.uniform(low=-1, high=1, size=N)

    # y ~ N(mu=f(x), std=0.2)
    mu = np.polyval(coeffs_true, x)
    std = 0.2 * np.random.randn(N)
    y = mu + std

    return x, y

# Let's try to fit a 4th degree polynomial to the data
# (more expressive hypothesis class).
# To incorpoate the constant bias, stack the features 
# so X is a matrix (N, 5),
# -> N = number of training examples
def make_features(N, degree=4):
    x, y = generate_data(N)
    X = np.column_stack([x**i for i in reversed(range(degree + 1))])
    return X, y

# sanity check plot
plt.figure(figsize=(4, 3))
N = 200
X, y = make_features(N)

plt.scatter(X[:, -2], y)
plt.xlabel("x")
plt.ylabel("y")

The trainable parameters of the NN:

In [ ]:
# Initialize the parameters (randomly) to have the desired shape
H = 16
d = X.shape[1]

"""
TO DO: Initialize the parameters
"""

# 1st layer of the NN
W1 = np.random.randn(H, d)
b1 = ... # fill in

# Now the 2nd layer of the NN
W2 = ...  # fill in
b2 = ...  # fill in

for v in [W1, b1, W2, b2]:
    print(v.shape)


Let's start implementing the building blocks

$$ReLU(x) = \max(x,0)$$

In [ ]:
def relu(x):
    return np.where(x>0, x, 0)


In [ ]:
z = np.array([1,3,-1])
relu(z)

In [ ]:
# Set up the feedforward model

# z = "W1x + b1"
z = np.einsum('hj,ij->ih',W1,X) + b1
print(z.shape)

h = relu(z)
print(h.shape)
y_pred = # fill in
print(y_pred.shape)

We're thinking about a regression task right now, so no need to have a non-linearity on the last layer.

In [ ]:
# Sanity check: Does the output have the dimensionality we expect?
assert y_pred.shape[0] == N

## Step 2b: Gradients of a NN

OK, let's now revise the NN with the gradient computation so we can do back prop. 

In [ ]:
def linear(W,X,b, grad=None):
    '''
    Linear layer mapping H = W x + b
    - W: array (m,n)
    - x: array (bs, n)
    - b: array (m)

    Outputs:
    - H: array (bs,m)

    If grad is an array (bs,m), return tuple: out, (grad_W, grad_b, grad_H)
    - grad_W: array (bs, m, n)
    - grad_b: array (bs, m)
    - grad_X: array (bs,    n)
    '''

    assert X.shape[1] == W.shape[1]
    assert W.shape[0] == b.shape[0]

    '''
    TO DO: Fill in
    '''
    raise NotImplementedError
    
    H = 
    
    if grad is not None:
        
        grad_W = 
        grad_b = 
        grad_X = 
        
        return H, (grad_W, grad_b, grad_X)
    else:
        return H

In [ ]:
def relu(z,grad=None):
    '''
    Non-linearity: better gradient flow
    '''
    out = 

    if grad is not None:
        '''
        To do: Calculate the local grad and return it too
        ''' 
        raise NotImplementedError 
    else:
        return out

In [ ]:
def myNN(X, param_dict, grad=None):
    '''
    Build an MLP with a single hidden layer.
    '''

    # Unpack the parameter dict
    W1, b1 = param_dict["W1"], param_dict["b1"]
    W2, b2 = param_dict["W2"], param_dict["b2"]

    # forward pass
    """
    TO DO: Fill in
    """

    if grad is not None:
        # reverse pass
        """
        TO DO: Fill in
        """

        grad_dict = {
            "W1": grad_W1.mean(axis=0),
            "b1": grad_b1.mean(axis=0),
            "W2": grad_W2.mean(axis=0),
            "b2": grad_b2.mean(axis=0),
        }

        return out.mean(), grad_dict
    else:
        return out.mean()

In [ ]:
param_dict ={'W1':W1,'b1':b1,'W2':W2,'b2':b2}

In [ ]:
# forward mode
out = myNN(X, param_dict)
out

In [ ]:
# reverse mode
dfdf = np.ones((1,1))
out, grad_dict = myNN(X, param_dict,np.ones((1,1)))
out

In [ ]:
for k in param_dict.keys():
    print(k,grad_dict[k].shape)

## Step 4: Cross check the gradients in jax

`Jax` is an **automatic differentiation** library where it can automatically keep track of the gradient propagation for us instead of us needing to keep track of everything manually with these "forward" and reverse modes.

We'll cover another auto diff library, `pytorch` in more detail tomorrow, but we'll use jax here for x-checking the gradients as it's a very transparent interface for these types of checks.

Also, it's super cute b/c the code stays _almost identical_, just replace `np` with `jnp`, so we can just reuse the forward pass NN code we wrote back in step 1.


In [ ]:
import jax
from jax import grad
from jax import numpy as jnp

Illustrative example: Calculate the gradient of :

$$g(x) = x^2, \qquad g'(x) = 2x$$

In [ ]:
def g(xi):
    return xi**2

In [ ]:
# Set up the gradient function
grad_g = grad(g) # 2x

# To evaluate the gradient, need to eval at a specific point, $x$
grad_g(3.)

In [ ]:
# Let's evaluate the same parameters... just need to type cast to numpy
param_dict_jax = {k:jnp.array(v) for k,v in param_dict.items()}

In [ ]:
def relu_jax(z):
    return jnp.where(z>0,z,0)

In [ ]:
def linear_jax(W,X,b):
    '''
    Linear layer mapping H = W x + b
    - W: array (m,n)
    - x: array (bs, n)
    - b: array (m)

    Outputs:
    - H: array (bs,m)
    '''
    
    return jnp.einsum('mn,bn->bm',W,X) + b

In [ ]:
# (X, param_dict)
def myNN_jax(param_dict):

    # Unpack the parameters
    W1 = param_dict['W1']
    b1 = param_dict['b1']
    
    W2 = param_dict['W2']
    b2 = param_dict['b2']

    # forward pass
    z1 = linear_jax(W1, X, b1)
    h1 = relu_jax(z1)

    z2 = linear_jax(W2, h1, b2)
    
    return z2.mean()

In [ ]:
grad_nn = grad(myNN_jax)

# And... evaluate it!
grad_dict_jax = grad_nn(param_dict_jax)

In [ ]:
for k, v in grad_dict_jax.items():
    print(k, v.shape, grad_dict[k].shape)

In [ ]:
for k, v in grad_dict_jax.items():
    print(k, np.all(np.isclose(v, grad_dict[k])))

^^ I give you all the jax code b/c the point of the problem is to sanity check your work, not become a jax expert (yet)!

I chose jax here b/c it's a very clean api for this type of check, but leaving as an exercise to you to cross-check the gradients in torch (it's a bit more syntax, but I show you how to do this in the lecture Tues morning!)

**Bonus:** Train the model  (on your own)

Awesome, we've played around with auto diff (AD), coded up both some intro examples and a simple NN, and cross checked these gradients with one of the autodiff packages on the market.

Tomorrow, we'll stick with these AD libraries  that do these "gutsy" bits for us, and move on to more architectures (CNNs, Deep Sets, transformers) as well as regularization tricks. But, having this detail oriented understanding is also great for building our inution for motivating architecture design choices.. and we'll highlight that tomorrow as well.